# Wind-tunnel Pressure Post-processing -- v3 paradigm

This notebook walks a typical consulting flow on the v3 paradigm:

1. Drop a body-pressure H5 and a reference-probe H5 into a directory.
2. Author four YAML pipeline templates (`cp.yaml`, `cf.yaml`, `cm.yaml`, `ce.yaml`).
3. Run them through `cfdmod run` (or the Python `run_template` API).
4. Inspect the per-body / per-zone time series and statistics.

Everything happens through composable ops; no recipe-specific Python entry points are needed.

For the conceptual intro see `notebooks/tutorials/`.

## 1. Imports

In [1]:
import pathlib
import shutil
import textwrap
import tempfile
import numpy as np
import h5py

from cfdmod import load_template, run_template
from cfdmod.adapters.xdmf_h5 import XdmfH5Storage

## 2. Set up a workspace from the galpao fixture

In production you'd point this at your case directory. The fixtures include a small wind-tunnel run we can use to demo end-to-end.

In [2]:
REPO_ROOT = pathlib.Path.cwd().parent  # adjust if running from elsewhere
FIXTURE = REPO_ROOT / "fixtures" / "tests" / "pressure"

tmpdir = pathlib.Path(tempfile.mkdtemp(prefix="cfdmod_demo_"))
(tmpdir / "templates").mkdir()
shutil.copytree(FIXTURE / "data", tmpdir / "data")
shutil.copytree(FIXTURE / "galpao", tmpdir / "galpao")
for tpl in (FIXTURE / "templates").iterdir():
    shutil.copy(tpl, tmpdir / "templates" / tpl.name)
print("workspace:", tmpdir)
for p in sorted(tmpdir.rglob("*")):
    rel = p.relative_to(tmpdir)
    if not p.is_dir() and len(rel.parts) <= 2:
        print("  ", rel)

workspace: /tmp/cfdmod_demo_n46urcym
   data/bodies.galpao.h5
   data/bodies.galpao.xdmf
   data/cp_t.normalized.h5
   data/cp_t.normalized.xdmf
   data/points.static_pressure.h5
   data/points.static_pressure.xdmf
   galpao/L1_xp.stl
   galpao/L2_yp.stl
   galpao/L3_zp_yp.stl
   galpao/L4_zp_ym.stl
   galpao/L5_ym.stl
   galpao/L6_xm.stl
   galpao/cfg.yaml
   galpao/galpao.lnas
   galpao/galpao.normalized.lnas
   galpao/galpao.normalized.stl
   galpao/m1_yp.stl
   galpao/m2_zp.stl
   galpao/m3_zm.stl
   galpao/p1_xp.stl
   galpao/p2_xp.stl
   galpao/p3_ym.stl
   galpao/p4_ym.stl
   galpao/p5_ym.stl
   galpao/p6c_yp.stl
   galpao/p7_xm.stl
   galpao/p8_yp.stl
   galpao/p9_yp.stl
   galpao/t1_ym.stl
   galpao/t2_yp.stl
   templates/ce.yaml
   templates/cf.yaml
   templates/cm.yaml
   templates/cp.yaml


## 3. Inspect the Cp template

Pipeline-as-YAML: three steps -- subtract reference, scale to Cp, compute statistics. Two outputs (timeseries + stats H5+XDMF pairs).

In [3]:
print((tmpdir / "templates" / "cp.yaml").read_text())

# Cp pipeline template (v3 schema).
#
# Reads body pressure and a static-pressure probe from disk, computes
# Cp = (p - p_ref) / dyn_pressure, optionally rescales time, and writes
# the time-resolved Cp series plus mean/rms/peak statistics.
#
# Anything you'd previously have configured under
# pressure_coefficient.default.statistics in cp_params.yaml now lives
# under the `statistics` step. Everything else is pipeline composition.
name: cp_default

inputs:
  body:
    kind: surface
    path: ../data/bodies.galpao
  p_ref:
    kind: points
    path: ../data/points.static_pressure

pipeline:
  # 1. Subtract the static reference per timestep (column-wise broadcast,
  #    rule 2). The result lives on a new "cp" field on the body source.
  - id: cp_unscaled
    kind: sub
    source: body
    rhs: p_ref
    field: pressure
    out: cp

  # 2. Divide by dynamic pressure. q_inf = 0.5 * rho * U_H^2
  #    Here U_H = 0.05, rho = 1.0 -> q = 0.00125. Scale = 1 / q.
  - id: cp_t
    kind: scale
  

## 4. Run the Cp pipeline

`run_template` returns the dict of every named binding the runner produced. Outputs declared in the YAML are also written to disk through the supplied `Storage`.

In [4]:
storage = XdmfH5Storage(pathlib.Path("/"))
cp_bindings = run_template(load_template(tmpdir / "templates" / "cp.yaml"), storage=storage)
print("bindings:", list(cp_bindings))

cp = cp_bindings["cp_t"]
stats = cp_bindings["cp_stats"]
print(f"\ncp_t  : {cp.kind}, {cp.n_elements} triangles, {cp.time.n_timesteps} steps")
print(f"stats : {stats.kind}, fields={stats.field_names}")
for f in stats.field_names:
    arr = stats.fields.read(f)
    print(f"  Cp {f:>4}: min={arr.min():+.3f}  max={arr.max():+.3f}")

bindings: ['body', 'p_ref', 'cp_unscaled', 'cp_t', 'cp_stats']

cp_t  : surface, 2915 triangles, 101 steps
stats : surface, fields=['mean', 'rms', 'min', 'max']
  Cp mean: min=-0.950  max=+0.455
  Cp  rms: min=+0.003  max=+0.308
  Cp  min: min=-1.173  max=+0.340
  Cp  max: min=-0.664  max=+0.758


## 5. Run Cf, Cm, Ce on top of the Cp time series

In [5]:
cf = run_template(load_template(tmpdir / "templates" / "cf.yaml"), storage=storage)
cm = run_template(load_template(tmpdir / "templates" / "cm.yaml"), storage=storage)
ce = run_template(load_template(tmpdir / "templates" / "ce.yaml"), storage=storage)

print("per-body Cf (mean over time):")
for d in ("x", "y", "z"):
    arr = cf[f"cf_{d}"].fields.read(f"cf_{d}")
    print(f"  cf_{d}: {arr.mean(axis=1)}")

print("\nper-body Cm (mean over time):")
for d in ("x", "y", "z"):
    arr = cm[f"cm_{d}"].fields.read(f"cm_{d}")
    print(f"  cm_{d}: {arr.mean(axis=1)}")

print("\nper-zone Ce (mean over time, one entry per non-empty zone):")
ce_arr = ce["ce"].fields.read("ce")
print("  zone means:", ce_arr.mean(axis=1))

per-body Cf (mean over time):
  cf_x: [-7.50435423]
  cf_y: [15.6650154]
  cf_z: [91.53279571]

per-body Cm (mean over time):
  cm_x: [87554.29851925]
  cm_y: [-135575.44322771]
  cm_z: [26091.18752234]

per-zone Ce (mean over time, one entry per non-empty zone):
  zone means: [-0.05459922  0.05876268 -0.39922781 -0.1870448  -0.04891312 -0.19892675
 -0.25423475 -0.44050057]


## 6. List the outputs on disk

In [6]:
out_dir = tmpdir / "templates" / "out"
for p in sorted(out_dir.iterdir()):
    print(" ", p.name, f" ({p.stat().st_size//1024} KB)")

  ce.time_series.h5  (2579 KB)
  ce.time_series.xdmf  (65 KB)
  cf_x.time_series.h5  (2579 KB)
  cf_x.time_series.xdmf  (66 KB)
  cf_y.time_series.h5  (2579 KB)
  cf_y.time_series.xdmf  (66 KB)
  cf_z.time_series.h5  (2579 KB)
  cf_z.time_series.xdmf  (66 KB)
  cm_x.time_series.h5  (2579 KB)
  cm_x.time_series.xdmf  (66 KB)
  cm_y.time_series.h5  (2579 KB)
  cm_y.time_series.xdmf  (66 KB)
  cm_z.time_series.h5  (2579 KB)
  cm_z.time_series.xdmf  (66 KB)
  cp.stats.h5  (242 KB)
  cp.stats.xdmf  (1 KB)
  cp.time_series.h5  (5082 KB)
  cp.time_series.xdmf  (85 KB)


## 7. Build your own pipeline

Templates are just YAML; you can chain extra ops (filters, custom statistics) by inserting steps. Example: add a moving-average smoothing before stats.

In [7]:
custom_yaml = textwrap.dedent(f"""
    name: cp_smoothed
    inputs:
      body:
        kind: surface
        path: {tmpdir}/data/bodies.galpao
      p_ref:
        kind: points
        path: {tmpdir}/data/points.static_pressure
    pipeline:
      - {{id: cp_unscaled, kind: sub,   source: body, rhs: p_ref, field: pressure, out: cp}}
      - {{id: cp_t,        kind: scale, source: cp_unscaled, field: cp, factor: 800.0}}
      - {{id: cp_smooth,   kind: moving_average, source: cp_t, field: cp,
         window: 3.0, out: cp_smooth}}
      - {{id: stats,       kind: statistics, source: cp_smooth, field: cp_smooth,
         kinds: [mean, rms, peak_min, peak_max]}}
    """).strip()

custom_yaml_path = tmpdir / "templates" / "cp_smoothed.yaml"
custom_yaml_path.write_text(custom_yaml)

out = run_template(load_template(custom_yaml_path), storage=storage)
smoothed = out["stats"].fields.read("rms")
raw = stats.fields.read("rms")
print(f"Cp rms (raw)      : mean = {raw.mean():.4f}")
print(f"Cp rms (smoothed) : mean = {smoothed.mean():.4f}")

Cp rms (raw)      : mean = 0.1540
Cp rms (smoothed) : mean = 0.1539


## 8. Clean up

In [8]:
shutil.rmtree(tmpdir)
print("removed", tmpdir)

removed /tmp/cfdmod_demo_n46urcym
